# Tenant Hazard Triage — fine-tune and evaluate  (notebook v4)

Settings: **Accelerator: GPU T4**, **Internet: on**, dataset **hazard-triage-bundle** added.
Then **Save Version → Save & Run All**. The log should start with `TRIAGE NOTEBOOK v4`.

Order of work:
1. Load the model (4-bit QLoRA) and attach an empty LoRA adapter (it still behaves exactly like the base model).
2. **Preflight (about 2 minutes):** run the longest prompts and one training step on the longest example.
   If memory or precision is going to be a problem, it fails here with a clear message, not hours later.
3. Score the **base** model on gold v2, the official scenarios and the synthetic val set.
4. Fine-tune on `train.jsonl` (loss on the answer only).
5. Score the **fine-tuned** model with the identical prompt and code.
6. Save predictions, logs and the adapter to the Output tab.

**Why Qwen3-4B-Instruct-2507:** the two bake-off models (Qwen3.5-4B, Gemma 4 E4B) can't run in float16
on a T4. Unsloth falls back to float32, which doubles memory and is very slow: Qwen3.5 was split across two
GPUs, and Gemma 4 ran out of memory in the first batch. Qwen3-4B (standard architecture, Apache 2.0) is
Unsloth's documented free-T4 path. Base and fine-tuned are scored here in the same harness, so the
comparison stays clean.

In [ ]:
# 1. Install (about 3-5 minutes)
!pip install -q --upgrade unsloth unsloth_zoo

In [ ]:
# 2. Settings
print("TRIAGE NOTEBOOK v4")
MODEL       = "unsloth/Qwen3-4B-Instruct-2507"
SHORT       = "qwen3-4b-2507"     # used in output file names
QLORA       = True
EPOCHS      = 2
LR          = 2e-4
LORA_R      = 16
MAX_LEN     = 2560                # prompt is ~1,900 tokens; longest example ~2,200
BATCH, ACCUM = 1, 8               # effective batch 8
EVAL_BATCH  = 4                   # halves automatically if memory runs out
RUN_BASE_EVAL = True
EXPORT_GGUF   = True              # optional, for running locally later; failures here don't matter
SEED = 3407

In [ ]:
# 3. GPU, environment and data
import glob, os, sys, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                          # one GPU: no slow cross-GPU splitting
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # less memory fragmentation
import torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4"
name = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
print(name, cap, f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
assert cap >= (7, 0), "This GPU is too old for Unsloth (P100). Switch the accelerator to T4."

hits = glob.glob("/kaggle/input/**/ft_lib.py", recursive=True)
assert hits, "Bundle not found: Add Input -> Your Datasets -> hazard-triage-bundle"
BUNDLE = os.path.dirname(hits[0]); sys.path.insert(0, BUNDLE)
OUT = "/kaggle/working"; os.chdir(OUT)
import ft_lib, spec
from evaluate import evaluate
print("bundle:", BUNDLE, "| spec", spec.SPEC_VERSION, "| prompt", ft_lib.PROMPT_HASH)

train_rows = ft_lib.read_jsonl(f"{BUNDLE}/train.jsonl")
val_rows   = ft_lib.read_jsonl(f"{BUNDLE}/val.jsonl")
TEST = {"gold_v2":   ft_lib.read_jsonl(f"{BUNDLE}/gold_v2.jsonl"),
        "official":  ft_lib.read_jsonl(f"{BUNDLE}/official.jsonl"),
        "synth_val": ft_lib.synth_val_as_gold(val_rows)}
ft_lib.write_jsonl(f"{OUT}/synth_val.jsonl", TEST["synth_val"])
print({k: len(v) for k, v in TEST.items()}, "| train", len(train_rows))

In [ ]:
# 4. Load the model (4-bit) and attach LoRA (adapter starts at zero = identical to the base model)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=QLORA, full_finetuning=False)
tok = getattr(tokenizer, "tokenizer", tokenizer)   # plain text tokenizer, even if a processor comes back
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

compute_dtype = model.get_input_embeddings().weight.dtype
print(f"compute dtype {compute_dtype} | GPU memory after load {torch.cuda.memory_allocated()/1e9:.1f} GB")
assert compute_dtype != torch.float32, (
    "This model is running in float32 on the T4 (no float16 support for its architecture). "
    "It will be too slow and run out of memory. Use the default MODEL in cell 2.")
MARK = ft_lib.end_marker(tok)
print("end-of-turn marker:", repr(MARK))
print(ft_lib.chat_text(tok, ft_lib.prompt_messages("TEST"), True)[-120:])

In [ ]:
# 5. Generation with automatic back-off: if a batch runs out of memory, it is split in half and retried
def make_gen(max_new_tokens=200):
    tok.padding_side = "left"
    def run(texts):
        prompts = [ft_lib.chat_text(tok, ft_lib.prompt_messages(t), True) for t in texts]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            seq = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        return tok.batch_decode(seq[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    def gen(texts):
        try:
            return run(texts)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(texts) == 1:
                raise
            h = len(texts) // 2
            print(f"    (out of memory at batch {len(texts)}; retrying as {h} + {len(texts)-h})")
            return gen(texts[:h]) + gen(texts[h:])
    return gen

# Preflight: the riskiest operations, on the largest inputs, before any long work starts
all_test = [r for rows in TEST.values() for r in rows]
longest_test = sorted(all_test, key=lambda r: len(r["text"]), reverse=True)[:EVAL_BATCH]
FastLanguageModel.for_inference(model)
t0 = time.time(); out = make_gen()( [r["text"] for r in longest_test] )
print(f"preflight generation OK ({time.time()-t0:.0f}s for {len(longest_test)} longest prompts)")
print("  sample answer:", out[0][:200].replace("\n", " "))

FastLanguageModel.for_training(model)
longest_train = max(train_rows, key=lambda r: len(r["text"]))
e = ft_lib.encode(tok, longest_train["text"], longest_train["output"], MAX_LEN, MARK)
assert e is not None, "Longest training example exceeds MAX_LEN; raise MAX_LEN in cell 2"
batch = {k: torch.tensor([v]).to(model.device) for k, v in e.items()}
torch.cuda.reset_peak_memory_stats()
loss = model(**batch).loss; loss.backward(); model.zero_grad(set_to_none=True)
del batch, loss; torch.cuda.empty_cache()
print(f"preflight training step OK ({len(e['input_ids'])} tokens, peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB)")

In [ ]:
# 6. Score every test set and write prediction files
def run_eval(tag):
    FastLanguageModel.for_inference(model)
    gen = make_gen()
    summary = {}
    for setname, rows in TEST.items():
        t0 = time.time()
        recs = ft_lib.predict(rows, gen, f"{tag}-{SHORT}", batch_size=EVAL_BATCH, log=lambda *a: None)
        ft_lib.write_jsonl(f"{OUT}/preds_{setname}_{tag}-{SHORT}.jsonl", recs)
        m = evaluate(rows, recs, f"{tag}-{SHORT}")["subsets"]["all"]
        h = m["headline"]
        summary[setname] = dict(emerg_recall=h["emergency_recall_strict"], emerg_precision=h["emergency_precision"],
                                review=h["human_review_rate"], sev_acc=m.get("severity_accuracy"),
                                parse_errors=sum(r["parse_error"] for r in recs))
        print(f"{tag} {setname:<10} ({len(rows)}) {time.time()-t0:5.0f}s  {summary[setname]}")
    json.dump(summary, open(f"{OUT}/summary_{tag}-{SHORT}.json", "w"), indent=2)
    FastLanguageModel.for_training(model)
    return summary

base = run_eval("base-hf") if RUN_BASE_EVAL else None

In [ ]:
# 7. Tokenise (loss on the answer only) and train
from datasets import Dataset
from transformers import Trainer, TrainingArguments

def build(rows):
    enc = [ft_lib.encode(tok, r["text"], r["output"], MAX_LEN, MARK) for r in rows]
    kept = [x for x in enc if x]
    print(f"  {len(kept)}/{len(rows)} examples (dropped {len(rows)-len(kept)} over {MAX_LEN} tokens)")
    return Dataset.from_list(kept)

train_ds, val_ds = build(train_rows), build(val_rows)
use_bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=f"{OUT}/ckpt", per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
    per_device_eval_batch_size=1, num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=10,
    lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.0, logging_steps=10,
    eval_strategy="epoch", save_strategy="no", fp16=not use_bf16, bf16=use_bf16,
    report_to="none", seed=SEED, remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=ft_lib.Collator(tok.pad_token_id))
t0 = time.time(); trainer.train()
print(f"trained in {(time.time()-t0)/60:.1f} min")
json.dump({"history": trainer.state.log_history, "minutes": (time.time()-t0)/60,
           "settings": dict(MODEL=MODEL, EPOCHS=EPOCHS, LR=LR, LORA_R=LORA_R, QLORA=QLORA, BATCH=BATCH, ACCUM=ACCUM,
                            MAX_LEN=MAX_LEN, train=len(train_ds), val=len(val_ds), prompt_hash=ft_lib.PROMPT_HASH)},
          open(f"{OUT}/train_log_{SHORT}.json", "w"), indent=2)

In [ ]:
# 8. Score the fine-tuned model (same prompt, same code), then compare
ft = run_eval("ft")
f = lambda v: "-" if v is None else (f"{v:.1%}" if isinstance(v, float) else str(v))
print(f"\n{'set':<10} {'metric':<16} {'base':>8} {'fine-tuned':>11}")
for s in ft:
    for k in ("emerg_recall", "emerg_precision", "review", "sev_acc", "parse_errors"):
        print(f"{s:<10} {k:<16} {f(base[s][k] if base else None):>8} {f(ft[s][k]):>11}")

In [ ]:
# 9. Save the adapter (small) — this is the trained model
model.save_pretrained(f"{OUT}/adapter_{SHORT}"); tok.save_pretrained(f"{OUT}/adapter_{SHORT}")
!cd /kaggle/working && zip -qr adapter_{SHORT}.zip adapter_{SHORT} && rm -rf adapter_{SHORT} ckpt
!ls -lh /kaggle/working

In [ ]:
# 10. Optional: GGUF for running locally (Ollama / llama.cpp). Failures here don't matter.
if EXPORT_GGUF:
    try:
        model.save_pretrained_gguf(f"{OUT}/gguf_{SHORT}", tok, quantization_method="q4_k_m")
        print("GGUF saved")
    except Exception as e:
        print("GGUF export skipped:", type(e).__name__, str(e)[:300])
!ls -lh /kaggle/working /kaggle/working/gguf_* 2>/dev/null

## Download from the Output tab
`preds_*.jsonl` (6 files), `synth_val.jsonl`, `summary_*.json` (2), `train_log_*.json`, `adapter_*.zip`
and, if it worked, the `.gguf` file.